# Silver Data: Web traffic + TF
Làm giống Products + EPD: đọc nguồn chính, kiểm tra từng lỗi của TF, đối chiếu các trường còn đúng rồi bỏ dòng TF đã có trong Web.
Cuối cùng so sánh tất cả các cột và chỉ thêm ngày mới hợp lệ. `date` là khóa chính.

In [146]:
import pandas as pd
import numpy as np

## Phần 1: Đọc và kiểm tra Web traffic

In [147]:
web = pd.read_csv('../web_traffic.csv')
display(web.head())
web.info()

,date,sessions,unique_visitors,page_views,bounce_rate,avg_session_duration_sec,traffic_source
0,2013-01-01,9760,7253,39093,0.00514,102.9,organic_search
1,2013-01-02,10456,8151,47611,0.00406,120.5,organic_search
2,2013-01-03,10076,7458,36963,0.00401,263.6,direct
3,2013-01-04,9973,8063,53078,0.00562,151.8,direct
4,2013-01-05,10223,7882,36790,0.00525,168.6,referral


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3652 entries, 0 to 3651
Data columns (total 7 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   date                      3652 non-null   object 
 1   sessions                  3652 non-null   int64  
 2   unique_visitors           3652 non-null   int64  
 3   page_views                3652 non-null   int64  
 4   bounce_rate               3652 non-null   float64
 5   avg_session_duration_sec  3652 non-null   float64
 6   traffic_source            3652 non-null   object 
dtypes: float64(2), int64(3), object(2)
memory usage: 199.8+ KB


In [148]:
# Kiểm tra null và trùng ngày
print(web.isna().sum())
print('Ngày có duy nhất không:', web['date'].is_unique)
display(web[web['date'].duplicated(keep=False)])

date                        0
sessions                    0
unique_visitors             0
page_views                  0
bounce_rate                 0
avg_session_duration_sec    0
traffic_source              0
dtype: int64
Ngày có duy nhất không: True


,date,sessions,unique_visitors,page_views,bounce_rate,avg_session_duration_sec,traffic_source


In [149]:
# Chuẩn hóa ngày và các cột số
web['date'] = pd.to_datetime(web['date'], format='%Y-%m-%d', errors='coerce')
number_cols = ['sessions', 'unique_visitors', 'page_views',
               'bounce_rate', 'avg_session_duration_sec']
for col in number_cols:
    web[col] = pd.to_numeric(web[col], errors='coerce')
web[number_cols] = web[number_cols].replace([np.inf, -np.inf], np.nan)

# Làm tròn để tránh sai số rất nhỏ khi so sánh CSV với JSON
web['bounce_rate'] = web['bounce_rate'].round(8)
web['avg_session_duration_sec'] = web['avg_session_duration_sec'].round(6)
web['traffic_source'] = web['traffic_source'].astype('string').str.strip().str.lower()
web['traffic_source'] = web['traffic_source'].replace('', pd.NA)

In [150]:
# Xem nguồn truy cập
valid_sources = ['direct', 'email_campaign', 'organic_search',
                 'paid_search', 'referral', 'social_media']
display(web['traffic_source'].value_counts(dropna=False))
display(web[~web['traffic_source'].isin(valid_sources)])

traffic_source
organic_search    1090
paid_search        784
social_media       632
email_campaign     505
referral           375
direct             266
Name: count, dtype: Int64

,date,sessions,unique_visitors,page_views,bounce_rate,avg_session_duration_sec,traffic_source


## Phần 2: Đọc và làm sạch TF

In [151]:
tf = pd.read_json('../tf.json', convert_dates=False)
display(tf.head())
tf.info()

,date,sessions,unique_visitors,page_views,bounce_rate,avg_session_duration_sec,traffic_source
0,2013-10-26,12422,8842,45914,0.00452,197.1,organic_search Variant 0001
1,2019-02-15,26260,19148,97929,0.00568,116.5,paid_search Variant 0002
2,2019-01-12,14383,11480,57737,0.00466,170.6,email_campaign Variant 0003
3,2017-09-30,22021,16969,95903,0.00344,111.8,paid_search Variant 0004
4,2021-08-12,38269,29771,208637,0.00442,210.7,paid_search Variant 0005


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 7 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   date                      1000 non-null   object 
 1   sessions                  999 non-null    object 
 2   unique_visitors           1000 non-null   object 
 3   page_views                1000 non-null   int64  
 4   bounce_rate               1000 non-null   float64
 5   avg_session_duration_sec  1000 non-null   float64
 6   traffic_source            1000 non-null   object 
dtypes: float64(2), int64(1), object(4)
memory usage: 54.8+ KB


In [152]:
# Chuyển ngày và số sang đúng kiểu; giá trị lỗi sẽ thành null
# Giữ bản gốc để xem lại nếu cần
tf_raw = tf.copy()
tf['date'] = pd.to_datetime(tf['date'], format='%Y-%m-%d', errors='coerce')
for col in number_cols:
    tf[col] = pd.to_numeric(tf[col], errors='coerce')
tf[number_cols] = tf[number_cols].replace([np.inf, -np.inf], np.nan)
tf['bounce_rate'] = tf['bounce_rate'].round(8)
tf['avg_session_duration_sec'] = tf['avg_session_duration_sec'].round(6)
print(tf.isna().sum())

date                        2
sessions                    2
unique_visitors             1
page_views                  0
bounce_rate                 0
avg_session_duration_sec    0
traffic_source              0
dtype: int64


In [153]:
# Chuẩn hóa khoảng trắng và chữ thường
tf['traffic_source'] = tf['traffic_source'].astype('string').str.strip().str.lower()
tf['traffic_source'] = tf['traffic_source'].str.replace(r'\s+', ' ', regex=True)

# Theo đối chiếu dữ liệu hiện tại, coi "Variant 0001" là nhãn phụ.
# Chỉ bỏ hậu tố nếu phần còn lại là một nguồn hợp lệ.
source = tf['traffic_source'].str.replace(r'\s+variant\s+\d+$', '', regex=True)
mask = source.isin(valid_sources)
tf.loc[mask, 'traffic_source'] = source[mask]
tf['traffic_source'] = tf['traffic_source'].replace(['', 'n/a', 'null', 'none'], pd.NA)
display(tf['traffic_source'].value_counts(dropna=False))

traffic_source
organic_search    302
paid_search       220
social_media      180
email_campaign    132
referral          102
direct             62
<NA>                2
Name: count, dtype: Int64

### Ngày bị thiếu hoặc sai

In [154]:
# Lấy các dòng cần kiểm tra
missing = tf[tf['date'].isna()].reset_index()
display(missing)

,index,date,sessions,unique_visitors,page_views,bounce_rate,avg_session_duration_sec,traffic_source
0,515,NaT,38996.0,31829.0,154237,0.00419,185.4,organic_search
1,728,NaT,16845.0,12989.0,54823,0.00338,269.9,email_campaign


In [155]:
# Đối chiếu các cột còn lại, bỏ qua date đang lỗi
cols = ['sessions', 'unique_visitors', 'page_views', 'bounce_rate', 'avg_session_duration_sec', 'traffic_source']
reference = web[cols].drop_duplicates()
matched = missing.merge(reference, on=cols, how='inner')
display(matched)

# Chỉ drop các dòng lỗi đã có bản tương ứng trong Web
# Dùng cột 'index' gốc, không dùng matched.index
print('Số dòng đã có trong Web và được bỏ:', matched['index'].nunique())
tf = tf.drop(index=matched['index'])

,index,date,sessions,unique_visitors,page_views,bounce_rate,avg_session_duration_sec,traffic_source
0,515,NaT,38996.0,31829.0,154237,0.00419,185.4,organic_search
1,728,NaT,16845.0,12989.0,54823,0.00338,269.9,email_campaign


Số dòng đã có trong Web và được bỏ: 2


### Sessions bị thiếu

In [156]:
# Lấy các dòng cần kiểm tra
missing = tf[tf['sessions'].isna()].reset_index()
display(missing)

,index,date,sessions,unique_visitors,page_views,bounce_rate,avg_session_duration_sec,traffic_source
0,31,2017-07-29,NaN,18977.0,137669,0.00502,108.7,paid_search
1,219,2016-08-17,NaN,22260.0,132681,0.00347,314.6,organic_search


In [157]:
# Đối chiếu các cột còn lại, bỏ qua sessions đang lỗi
cols = ['date', 'unique_visitors', 'page_views', 'bounce_rate', 'avg_session_duration_sec', 'traffic_source']
reference = web[cols].drop_duplicates()
matched = missing.merge(reference, on=cols, how='inner')
display(matched)

# Chỉ drop các dòng lỗi đã có bản tương ứng trong Web
# Dùng cột 'index' gốc, không dùng matched.index
print('Số dòng đã có trong Web và được bỏ:', matched['index'].nunique())
tf = tf.drop(index=matched['index'])

,index,date,sessions,unique_visitors,page_views,bounce_rate,avg_session_duration_sec,traffic_source
0,31,2017-07-29,NaN,18977.0,137669,0.00502,108.7,paid_search
1,219,2016-08-17,NaN,22260.0,132681,0.00347,314.6,organic_search


Số dòng đã có trong Web và được bỏ: 2


### Sessions bị âm

In [158]:
# Lấy các dòng cần kiểm tra
missing = tf[tf['sessions'] < 0].reset_index()
display(missing)

,index,date,sessions,unique_visitors,page_views,bounce_rate,avg_session_duration_sec,traffic_source
0,107,2013-06-21,-100.0,21451.0,103139,0.00566,139.1,email_campaign


In [159]:
# Đối chiếu các cột còn lại, bỏ qua sessions đang lỗi
cols = ['date', 'unique_visitors', 'page_views', 'bounce_rate', 'avg_session_duration_sec', 'traffic_source']
reference = web[cols].drop_duplicates()
matched = missing.merge(reference, on=cols, how='inner')
display(matched)

# Chỉ drop các dòng lỗi đã có bản tương ứng trong Web
# Dùng cột 'index' gốc, không dùng matched.index
print('Số dòng đã có trong Web và được bỏ:', matched['index'].nunique())
tf = tf.drop(index=matched['index'])

,index,date,sessions,unique_visitors,page_views,bounce_rate,avg_session_duration_sec,traffic_source
0,107,2013-06-21,-100.0,21451.0,103139,0.00566,139.1,email_campaign


Số dòng đã có trong Web và được bỏ: 1


### Sessions lớn hơn page_views

In [160]:
# Lấy các dòng cần kiểm tra
missing = tf[tf['sessions'] > tf['page_views']].reset_index()
display(missing)

,index,date,sessions,unique_visitors,page_views,bounce_rate,avg_session_duration_sec,traffic_source
0,421,2014-09-29,999999999.0,14785.0,75135,0.00411,136.3,referral


In [161]:
# Đối chiếu các cột còn lại, bỏ qua sessions đang lỗi
cols = ['date', 'unique_visitors', 'page_views', 'bounce_rate', 'avg_session_duration_sec', 'traffic_source']
reference = web[cols].drop_duplicates()
matched = missing.merge(reference, on=cols, how='inner')
display(matched)

# Chỉ drop các dòng lỗi đã có bản tương ứng trong Web
# Dùng cột 'index' gốc, không dùng matched.index
print('Số dòng đã có trong Web và được bỏ:', matched['index'].nunique())
tf = tf.drop(index=matched['index'])

,index,date,sessions,unique_visitors,page_views,bounce_rate,avg_session_duration_sec,traffic_source
0,421,2014-09-29,999999999.0,14785.0,75135,0.00411,136.3,referral


Số dòng đã có trong Web và được bỏ: 1


### Unique visitors bị thiếu

In [162]:
# Lấy các dòng cần kiểm tra
missing = tf[tf['unique_visitors'].isna()].reset_index()
display(missing)

,index,date,sessions,unique_visitors,page_views,bounce_rate,avg_session_duration_sec,traffic_source
0,633,2015-11-20,13750.0,NaN,61102,0.00364,313.1,paid_search


In [163]:
# Đối chiếu các cột còn lại, bỏ qua unique_visitors đang lỗi
cols = ['date', 'sessions', 'page_views', 'bounce_rate', 'avg_session_duration_sec', 'traffic_source']
reference = web[cols].drop_duplicates()
matched = missing.merge(reference, on=cols, how='inner')
display(matched)

# Chỉ drop các dòng lỗi đã có bản tương ứng trong Web
# Dùng cột 'index' gốc, không dùng matched.index
print('Số dòng đã có trong Web và được bỏ:', matched['index'].nunique())
tf = tf.drop(index=matched['index'])

,index,date,sessions,unique_visitors,page_views,bounce_rate,avg_session_duration_sec,traffic_source
0,633,2015-11-20,13750.0,NaN,61102,0.00364,313.1,paid_search


Số dòng đã có trong Web và được bỏ: 1


### Unique visitors bị âm

In [164]:
# Lấy các dòng cần kiểm tra
missing = tf[tf['unique_visitors'] < 0].reset_index()
display(missing)

,index,date,sessions,unique_visitors,page_views,bounce_rate,avg_session_duration_sec,traffic_source
0,306,2022-11-09,14299.0,-50.0,78119,0.00423,314.2,email_campaign


In [165]:
# Đối chiếu các cột còn lại, bỏ qua unique_visitors đang lỗi
cols = ['date', 'sessions', 'page_views', 'bounce_rate', 'avg_session_duration_sec', 'traffic_source']
reference = web[cols].drop_duplicates()
matched = missing.merge(reference, on=cols, how='inner')
display(matched)

# Chỉ drop các dòng lỗi đã có bản tương ứng trong Web
# Dùng cột 'index' gốc, không dùng matched.index
print('Số dòng đã có trong Web và được bỏ:', matched['index'].nunique())
tf = tf.drop(index=matched['index'])

,index,date,sessions,unique_visitors,page_views,bounce_rate,avg_session_duration_sec,traffic_source
0,306,2022-11-09,14299.0,-50.0,78119,0.00423,314.2,email_campaign


Số dòng đã có trong Web và được bỏ: 1


### Nguồn truy cập bị thiếu hoặc không hợp lệ

In [166]:
# Lấy các dòng cần kiểm tra
missing = tf[~tf['traffic_source'].isin(valid_sources)].reset_index()
display(missing)

,index,date,sessions,unique_visitors,page_views,bounce_rate,avg_session_duration_sec,traffic_source
0,744,2020-05-23,43072.0,32884.0,203926,0.00343,169.8,<NA>
1,934,2016-01-15,12653.0,10155.0,55447,0.00452,290.1,<NA>


In [167]:
# Đối chiếu các cột còn lại, bỏ qua traffic_source đang lỗi
cols = ['date', 'sessions', 'unique_visitors', 'page_views', 'bounce_rate', 'avg_session_duration_sec']
reference = web[cols].drop_duplicates()
matched = missing.merge(reference, on=cols, how='inner')
display(matched)

# Chỉ drop các dòng lỗi đã có bản tương ứng trong Web
# Dùng cột 'index' gốc, không dùng matched.index
print('Số dòng đã có trong Web và được bỏ:', matched['index'].nunique())
tf = tf.drop(index=matched['index'])

,index,date,sessions,unique_visitors,page_views,bounce_rate,avg_session_duration_sec,traffic_source
0,744,2020-05-23,43072.0,32884.0,203926,0.00343,169.8,<NA>
1,934,2016-01-15,12653.0,10155.0,55447,0.00452,290.1,<NA>


Số dòng đã có trong Web và được bỏ: 2


In [168]:
# Bỏ dòng trùng hoàn toàn, kiểm tra các dòng trùng ngày còn lại
print('TF trùng toàn bộ:', tf.duplicated().sum())
tf = tf.drop_duplicates()
duplicate_dates = tf[tf['date'].duplicated(keep=False)].copy()
display(duplicate_dates)

# Giữ riêng các dòng trùng ngày nhưng khác số liệu để kiểm tra
tf = tf.drop(index=duplicate_dates.index)
tf = tf.reset_index(drop=True)
print('TF còn lại để so sánh:', len(tf))

TF trùng toàn bộ: 0


,date,sessions,unique_visitors,page_views,bounce_rate,avg_session_duration_sec,traffic_source


TF còn lại để so sánh: 990


## Phần 3: Merge và kiểm tra dòng mới

In [169]:
# So sánh tất cả các cột
compare_cols = ['date', 'sessions', 'unique_visitors', 'page_views',
                'bounce_rate', 'avg_session_duration_sec', 'traffic_source']
merged = tf.merge(
    web[compare_cols].drop_duplicates(),
    on=compare_cols, how='left', indicator=True
)

# both: đã có trong Web; left_only: chưa khớp toàn bộ
already_exists = merged[merged['_merge'] == 'both']
not_matched = merged[merged['_merge'] == 'left_only'].drop(columns='_merge')

print('TF đã có trong Web:', len(already_exists))
print('TF chưa khớp:', len(not_matched))
display(not_matched)

TF đã có trong Web: 990
TF chưa khớp: 0


,date,sessions,unique_visitors,page_views,bounce_rate,avg_session_duration_sec,traffic_source


In [170]:
# Chưa khớp nhưng cùng ngày: có số liệu khác, giữ Web làm nguồn chính
conflicts = not_matched[not_matched['date'].isin(web['date'])].copy()
print('Cùng ngày nhưng khác số liệu:', len(conflicts))
display(conflicts.merge(web, on='date', suffixes=('_tf', '_web')))

# Chỉ thêm những ngày chưa tồn tại trong Web
new_from_tf = not_matched[~not_matched['date'].isin(web['date'])].copy()
print('Ngày mới được thêm:', len(new_from_tf))
display(new_from_tf)

Cùng ngày nhưng khác số liệu: 0


,date,sessions_tf,unique_visitors_tf,page_views_tf,bounce_rate_tf,avg_session_duration_sec_tf,traffic_source_tf,sessions_web,unique_visitors_web,page_views_web,bounce_rate_web,avg_session_duration_sec_web,traffic_source_web


Ngày mới được thêm: 0


,date,sessions,unique_visitors,page_views,bounce_rate,avg_session_duration_sec,traffic_source


In [171]:
# Gộp và kiểm tra kết quả cuối cùng
traffic_silver = pd.concat([web, new_from_tf], ignore_index=True)
traffic_silver = traffic_silver.sort_values('date').reset_index(drop=True)
for col in ['sessions', 'unique_visitors', 'page_views']:
    traffic_silver[col] = traffic_silver[col].astype('Int64')

print('Tổng số dòng:', len(traffic_silver))
print('Số ngày trùng:', traffic_silver['date'].duplicated().sum())
print('Số null:', traffic_silver.isna().sum().sum())
traffic_silver.info()

Tổng số dòng: 3652
Số ngày trùng: 0
Số null: 0
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3652 entries, 0 to 3651
Data columns (total 7 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   date                      3652 non-null   datetime64[ns]
 1   sessions                  3652 non-null   Int64         
 2   unique_visitors           3652 non-null   Int64         
 3   page_views                3652 non-null   Int64         
 4   bounce_rate               3652 non-null   float64       
 5   avg_session_duration_sec  3652 non-null   float64       
 6   traffic_source            3652 non-null   string        
dtypes: Int64(3), datetime64[ns](1), float64(2), string(1)
memory usage: 210.5 KB


In [172]:

traffic_silver.to_csv('../SilverData/web_traffic_silver.csv', index=False)
  